# NB01 — Barren Plateaus and QMKL Recovery

**Research question:** When does increasing qubit count $Q$ hurt performance, and how does QMKL (multiple small-$Q$ kernels) fix it?

## What this notebook proves
- **Section A:** Gradient variance decays exponentially with $Q$ → barren plateaus are real
- **Section B:** Kernel concentration grows with $Q$ → all pairs become indistinguishable
- **Section C:** Single-kernel AUC peaks around $Q \approx 4$–6 and degrades for large $Q$
- **Section D:** QMKL with $M$ small kernels recovers performance — and surpasses the best single kernel

All computation is analytical (numpy). No Qiskit required.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from src.data.loaders import load_fred_recession_synthetic
from src.preprocessing.scaler import QuantumScaler
from src.kernels.analytical import K_Z, K_ZZ, K_XZ
from src.kernels.subset_kernels import build_subset_kernels_train_test, non_overlapping_subsets
from src.kernels.diagnostics import gradient_variance, kernel_concentration_std
from src.mkl.combiner import MultipleKernelCombiner

plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 130})

X_all, y_all, feats = load_fred_recession_synthetic(n_samples=300, seed=42)
X_all = QuantumScaler().fit_transform(X_all)
print(f'Dataset: {X_all.shape}, recession rate: {y_all.mean():.1%}')

## Section A — Gradient Variance vs Q

The **barren plateau** theorem (McClean et al., 2018) states that for random quantum circuits:
$$\text{Var}\left[\frac{\partial K}{\partial \alpha}\right] \leq \frac{C}{2^Q}$$

We verify this empirically: as $Q$ grows, the gradient of $K_{ZZ}$ w.r.t. the bandwidth $\alpha$ vanishes exponentially. This makes gradient-based kernel training impossible at large $Q$.

In [ ]:
Q_range_A = [2, 4, 6, 8, 10, 12, 14, 16]
N_SEEDS = 10
N_SAMPLES_A = 60  # keep fast

grad_var_mean = []
grad_var_std = []

for Q in Q_range_A:
    vars_q = []
    for seed in range(N_SEEDS):
        rng = np.random.RandomState(seed)
        # Pick Q random features
        feat_idx = rng.choice(X_all.shape[1], size=min(Q, X_all.shape[1]), replace=False)
        X_sub = X_all[:N_SAMPLES_A, feat_idx]
        v = gradient_variance(X_sub, K_ZZ, alpha=1.0, epsilon=0.01)
        vars_q.append(v)
    grad_var_mean.append(np.mean(vars_q))
    grad_var_std.append(np.std(vars_q))
    print(f'  Q={Q:2d}: var={np.mean(vars_q):.3e} ± {np.std(vars_q):.3e}')

In [ ]:
grad_var_mean = np.array(grad_var_mean)
grad_var_std = np.array(grad_var_std)
Q_arr = np.array(Q_range_A)

# Fit exponential decay: log(var) = a + b*Q
log_var = np.log10(np.clip(grad_var_mean, 1e-20, None))
valid = np.isfinite(log_var)
coeffs_A = np.polyfit(Q_arr[valid], log_var[valid], 1)
Q_fit = np.linspace(Q_arr[0], Q_arr[-1], 100)
log_var_fit = np.polyval(coeffs_A, Q_fit)

print(f'Fit: log10(Var) = {coeffs_A[0]:.4f}*Q + {coeffs_A[1]:.4f}')
print(f'Decay per qubit: ×{10**coeffs_A[0]:.4f} (expected ≈ 0.5 for barren plateau)')

## Section B — Kernel Concentration vs Q

A **concentrated** kernel has $\text{std}(K_{ij}) \approx 0$ for $i \neq j$:
all data pairs become indistinguishable. The kernel loses discriminative power.

For $K_{ZZ}$, this scales as $\sim 2^{-Q/2}$ — the same exponential as the gradient variance.

In [ ]:
Q_range_B = [2, 4, 6, 8, 10, 12, 14, 16]
N_SAMPLES_B = 60

conc_mean = []
conc_std = []

for Q in Q_range_B:
    stds_q = []
    for seed in range(N_SEEDS):
        rng = np.random.RandomState(seed)
        feat_idx = rng.choice(X_all.shape[1], size=min(Q, X_all.shape[1]), replace=False)
        X_sub = X_all[:N_SAMPLES_B, feat_idx]
        K = K_ZZ(X_sub, X_sub)
        stds_q.append(kernel_concentration_std(K))
    conc_mean.append(np.mean(stds_q))
    conc_std.append(np.std(stds_q))
    print(f'  Q={Q:2d}: std(K_off_diag)={np.mean(stds_q):.4f} ± {np.std(stds_q):.4f}')

In [ ]:
conc_mean = np.array(conc_mean)
conc_std = np.array(conc_std)

# Fit exponential decay on log scale
log_conc = np.log10(np.clip(conc_mean, 1e-15, None))
valid_b = np.isfinite(log_conc)
coeffs_B = np.polyfit(Q_arr[valid_b], log_conc[valid_b], 1)
log_conc_fit = np.polyval(coeffs_B, Q_fit)

print(f'Fit: log10(std) = {coeffs_B[0]:.4f}*Q + {coeffs_B[1]:.4f}')
print(f'Effective halving every {-np.log10(2)/coeffs_B[0]:.1f} qubits')

## Section C — AUC vs Q (Single Kernel)

Despite the theoretical problem, **does it hurt in practice?**

We train SVC with $K_{ZZ}$ using only the first $Q$ features. For small $Q$, the kernel is
expressive but ignores most features. For large $Q$, all entries concentrate and the SVM
cannot find a decision boundary.

Prediction: AUC peaks at intermediate $Q$ and degrades for $Q \gtrsim 8$.

In [ ]:
Q_range_C = [2, 4, 6, 8, 10, 12, 14, 16, 19]
N_CV = 5

auc_qzz_mean = []
auc_qzz_std = []

skf = StratifiedKFold(n_splits=N_CV, shuffle=True, random_state=0)

for Q in Q_range_C:
    X_q = X_all[:, :Q]
    aucs = []
    for tr, te in skf.split(X_q, y_all):
        X_tr, X_te = X_q[tr], X_q[te]
        y_tr, y_te = y_all[tr], y_all[te]
        K_tr = K_ZZ(X_tr, X_tr)
        K_te = K_ZZ(X_te, X_tr)
        clf = SVC(kernel='precomputed', C=1.0, probability=True)
        clf.fit(K_tr, y_tr)
        scores = clf.predict_proba(K_te)[:, 1]
        aucs.append(roc_auc_score(y_te, scores))
    auc_qzz_mean.append(np.mean(aucs))
    auc_qzz_std.append(np.std(aucs))
    print(f'  Q={Q:2d}: AUC={np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

In [ ]:
# RBF-SVM baseline (classical, all features)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler as SS

rbf_aucs = []
for tr, te in skf.split(X_all, y_all):
    clf_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
    clf_rbf.fit(X_all[tr], y_all[tr])
    scores = clf_rbf.predict_proba(X_all[te])[:, 1]
    rbf_aucs.append(roc_auc_score(y_all[te], scores))

rbf_auc = np.mean(rbf_aucs)
rbf_auc_std = np.std(rbf_aucs)
print(f'RBF-SVM (all {X_all.shape[1]} features): AUC={rbf_auc:.4f} ± {rbf_auc_std:.4f}')

auc_qzz_mean = np.array(auc_qzz_mean)
auc_qzz_std = np.array(auc_qzz_std)
best_single_auc = auc_qzz_mean.max()
best_single_Q = Q_range_C[auc_qzz_mean.argmax()]
print(f'Best single-kernel AUC: {best_single_auc:.4f} at Q={best_single_Q}')

## Section D — QMKL Recovery

**Key insight:** instead of one large kernel, use $M$ non-overlapping kernels each with $Q=4$ qubits.
With $d=19$ features and $M=4$ kernels: we cover 16 features, each kernel stays in the
non-concentrated regime.

As $M$ grows, coverage improves and AUC increases — until we hit the plateau where
all useful features are covered.

In [ ]:
Q_fixed = 4
d = X_all.shape[1]  # 19
M_range = [1, 2, 3, 4, 5, 7]

auc_mkl_mean = []
auc_mkl_std = []

for M in M_range:
    assignment = non_overlapping_subsets(d=d, Q=Q_fixed, M=M)
    aucs = []
    for tr, te in skf.split(X_all, y_all):
        X_tr, X_te = X_all[tr], X_all[te]
        y_tr, y_te = y_all[tr], y_all[te]
        K_trains, K_tests = build_subset_kernels_train_test(
            X_tr, X_te, assignment,
            kernel_configs=[('ZZ', 1.0)] * M
        )
        combiner = MultipleKernelCombiner('centered')
        K_tr_combined = combiner.fit_combine(K_trains, y_tr)
        K_te_combined = combiner.combine(K_tests)
        clf = SVC(kernel='precomputed', C=1.0, probability=True)
        clf.fit(K_tr_combined, y_tr)
        scores = clf.predict_proba(K_te_combined)[:, 1]
        aucs.append(roc_auc_score(y_te, scores))
    auc_mkl_mean.append(np.mean(aucs))
    auc_mkl_std.append(np.std(aucs))
    print(f'  M={M}: AUC={np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

auc_mkl_mean = np.array(auc_mkl_mean)
auc_mkl_std = np.array(auc_mkl_std)

## Final Figure — 2×2 Summary

Combines all four results into one publication-ready figure.

In [ ]:
import os
os.makedirs('../results/figures', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Barren Plateaus & QMKL Recovery — FRED Recession Dataset',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel A: Gradient variance ────────────────────────────────────────────────
ax = axes[0, 0]
ax.errorbar(Q_arr, np.log10(np.clip(grad_var_mean, 1e-20, None)),
            yerr=grad_var_std / (grad_var_mean * np.log(10) + 1e-20),
            fmt='o-', color='#e74c3c', capsize=4, label='Empirical (K_ZZ)')
ax.plot(Q_fit, log_var_fit, '--', color='#c0392b', alpha=0.7,
        label=f'Fit: slope={coeffs_A[0]:.3f}/qubit')
ax.set_xlabel('Number of qubits Q')
ax.set_ylabel('log₁₀ Var[dK/dα]')
ax.set_title('(A) Barren Plateau: Gradient Variance')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.05, 'Exponential decay\n→ gradient-free training', transform=ax.transAxes,
        fontsize=8, color='#c0392b', va='bottom')



In [ ]:
# ── Panel B: Kernel concentration ────────────────────────────────────────────
ax = axes[0, 1]
ax.errorbar(Q_arr, np.log10(np.clip(conc_mean, 1e-15, None)),
            yerr=conc_std / (conc_mean * np.log(10) + 1e-15),
            fmt='s-', color='#3498db', capsize=4, label='std(K_off_diag)')
ax.plot(Q_fit, log_conc_fit, '--', color='#2980b9', alpha=0.7,
        label=f'Fit: slope={coeffs_B[0]:.3f}/qubit')
ax.set_xlabel('Number of qubits Q')
ax.set_ylabel('log₁₀ std(K[i,j], i≠j)')
ax.set_title('(B) Kernel Concentration')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.05, 'std→0 ⟹ all pairs\nindistinguishable', transform=ax.transAxes,
        fontsize=8, color='#2980b9', va='bottom')


In [ ]:

# ── Panel C: AUC vs Q (single kernel) ────────────────────────────────────────
ax = axes[1, 0]
ax.errorbar(Q_range_C, auc_qzz_mean, yerr=auc_qzz_std,
            fmt='D-', color='#2ecc71', capsize=4, label='K_ZZ single kernel')
ax.axhline(rbf_auc, color='#95a5a6', ls='--', lw=1.5, label=f'RBF-SVM (AUC={rbf_auc:.3f})')
ax.axvline(best_single_Q, color='#2ecc71', ls=':', alpha=0.6,
           label=f'Best Q={best_single_Q}')
ax.set_xlabel('Number of qubits Q')
ax.set_ylabel('5-fold CV AUC')
ax.set_title('(C) AUC vs Q — Single Kernel')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.4, 1.0])



In [ ]:
# ── Panel D: QMKL recovery ────────────────────────────────────────────────────
ax = axes[1, 1]
ax.errorbar(M_range, auc_mkl_mean, yerr=auc_mkl_std,
            fmt='o-', color='#9b59b6', capsize=4, lw=2, label='QMKL (Q=4, centered)')
ax.axhline(best_single_auc, color='#2ecc71', ls='--', lw=1.5,
           label=f'Best single K_ZZ (AUC={best_single_auc:.3f})')
ax.axhline(rbf_auc, color='#95a5a6', ls='--', lw=1.5,
           label=f'RBF-SVM (AUC={rbf_auc:.3f})')
ax.set_xlabel('Number of kernels M (Q=4 each)')
ax.set_ylabel('5-fold CV AUC')
ax.set_title('(D) QMKL Recovery vs M')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.4, 1.0])

plt.tight_layout()
save_path = '../results/figures/NB01_barren_plateau.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Figure saved to {save_path}')
plt.show()

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
print('=== Summary ===')
print(f'Dataset: FRED Recession, N={X_all.shape[0]}, d={X_all.shape[1]}')
print()
print('Barren plateau (Section A):')
print(f'  Gradient var decay: {coeffs_A[0]:.4f} log10-units per qubit')
print(f'  At Q=16: var ≈ {10**np.polyval(coeffs_A, 16):.2e} (vs Q=2: {10**np.polyval(coeffs_A, 2):.2e})')
print()
print('Kernel concentration (Section B):')
print(f'  std(K_offdiag) decay: {coeffs_B[0]:.4f} log10-units per qubit')
print()
print('Single kernel AUC (Section C):')
for Q, auc, s in zip(Q_range_C, auc_qzz_mean, auc_qzz_std):
    print(f'  Q={Q:2d}: {auc:.4f} ± {s:.4f}')
print(f'  RBF baseline: {rbf_auc:.4f} ± {rbf_auc_std:.4f}')
print()
print('QMKL recovery (Section D, Q=4 each kernel):')
for M, auc, s in zip(M_range, auc_mkl_mean, auc_mkl_std):
    delta = auc - best_single_auc
    print(f'  M={M}: {auc:.4f} ± {s:.4f}  (Δ vs best single: {delta:+.4f})')